# MOJAVE centering heuristics

Run the notebook from top to bottom. It loads preprocessed `.icn.pt` images, estimates
background from outside a central rectangle, and compares simple centering rules.
The DataFrame stores file paths and measurements; the image pixels are loaded again
only for examples you choose to plot.


In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from tqdm.auto import tqdm
from IPython.display import display

import os
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context

## Dataset settings

Set all data, heuristic, and plotting values here. Use `SAMPLE_LIMIT = None` to process every file.

In [ ]:
DATA_DIR = Path("/cephfs/users/aaki/data/Mojave_I_256")
SAMPLE_LIMIT = None
SEED = 42

## Load and define the masks

The rectangle marks the central area. Its inverse estimates the background; the
threshold then selects candidate emission pixels across the whole image.


In [ ]:
def load_image(path):
    image = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(image, torch.Tensor):
        raise ValueError("Expected a saved tensor")
    if image.ndim == 3 and image.shape[0] == 1:
        image = image[0]
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image or 1×H×W tensor, got {tuple(image.shape)}")
    if image.is_complex():
        raise ValueError("Complex images are not supported")
    return image.to(torch.float32).numpy()

def rectangular_mask(shape, fraction):
    height, width = shape
    box_height = int(height * fraction)
    box_width = int(width * fraction)
    y0 = (height - box_height) // 2
    x0 = (width - box_width) // 2

    mask = np.zeros(shape, dtype=bool)
    mask[y0:y0 + box_height, x0:x0 + box_width] = True
    return mask

def estimate_background(image, background_mask):
    values = image[background_mask & np.isfinite(image)]
    if values.size == 0:
        raise ValueError("No finite pixels available for the background estimate")

    level = float(np.median(values))
    spread = float(1.4826 * np.median(np.abs(values - level)))
    return level, spread

def candidate_mask(image, background_level, background_spread, sigma_multiplier):
    threshold = background_level + sigma_multiplier * background_spread
    mask = np.isfinite(image) & (image > threshold)
    return mask, float(threshold)

## Measure centering in several ways

Peak, binary-centroid, and weighted-centroid locations use thresholded candidate pixels. The 95% rule uses all positive background-subtracted flux in the image. Distance rules use the tolerance below; missing emission or zero total flux is marked unknown.

In [ ]:
def centering_metrics(image, emission_mask, rectangle_mask, background_level,
                      tolerance_fraction, required_mass_fraction):
    height, width = image.shape
    center_x, center_y = (width - 1) / 2, (height - 1) / 2
    tolerance = tolerance_fraction * min(height, width)
    y, x = np.where(emission_mask)

    result = {
        "center_x": center_x,
        "center_y": center_y,
        "tolerance_px": tolerance,
    }
    if len(x) == 0:
        result.update({
            "peak_x": np.nan, "peak_y": np.nan, "peak_distance_px": np.nan,
            "peak_centered": None,
            "binary_x": np.nan, "binary_y": np.nan, "binary_distance_px": np.nan,
            "binary_centered": None,
            "weighted_x": np.nan, "weighted_y": np.nan, "weighted_distance_px": np.nan,
            "weighted_centered": None,
            "mass_fraction_in_rectangle": np.nan, "mass_95pct_centered": None,
        })
        return result

    # Brightest candidate pixel.
    peak_index = np.argmax(image[y, x])
    peak_x, peak_y = int(x[peak_index]), int(y[peak_index])

    # Unweighted center of all candidate pixels.
    binary_x, binary_y = float(x.mean()), float(y.mean())

    # Centroid weighted by positive, background-subtracted intensity.
    weights = np.maximum(image[y, x] - background_level, 0)
    weighted_x = float(np.average(x, weights=weights))
    weighted_y = float(np.average(y, weights=weights))

    def distance(x_position, y_position):
        return float(np.hypot(x_position - center_x, y_position - center_y))

    peak_distance = distance(peak_x, peak_y)
    binary_distance = distance(binary_x, binary_y)
    weighted_distance = distance(weighted_x, weighted_y)

    # Positive excess flux over the whole image, before thresholding to candidates.
    flux = np.where(np.isfinite(image), np.maximum(image - background_level, 0), 0)
    total_flux = float(flux.sum())
    inside_flux = float(flux[rectangle_mask].sum())
    mass_fraction = inside_flux / total_flux if total_flux > 0 else np.nan

    result.update({
        "peak_x": peak_x, "peak_y": peak_y, "peak_distance_px": peak_distance,
        "peak_centered": peak_distance <= tolerance,
        "binary_x": binary_x, "binary_y": binary_y, "binary_distance_px": binary_distance,
        "binary_centered": binary_distance <= tolerance,
        "weighted_x": weighted_x, "weighted_y": weighted_y,
        "weighted_distance_px": weighted_distance, "weighted_centered": weighted_distance <= tolerance,
        "mass_fraction_in_rectangle": mass_fraction,
        "mass_95pct_centered": (mass_fraction >= required_mass_fraction
                                 if np.isfinite(mass_fraction) else None),
    })
    return result


## Settings for analysis loop

In [ ]:
RECTANGLE_FRACTION = 0.5
BACKGROUND_SIGMA_MULTIPLIER = 3.0
CENTER_TOLERANCE_FRACTION = 0.05
MASS_FRACTION_REQUIRED = 0.95

## Run one analysis loop

The loop reports progress and stores one row of scalar results and a
file path per image.


In [ ]:
MAX_WORKERS = min(20, os.cpu_count() or 1)


def analyze_one(path):
    row = {"filename": path.name, "path": str(path), "status": "ok"}

    try:
        image = load_image(path)
        rectangle = rectangular_mask(image.shape, RECTANGLE_FRACTION)
        background_level, background_spread = estimate_background(image, ~rectangle)
        emission, threshold = candidate_mask(
            image, background_level, background_spread, BACKGROUND_SIGMA_MULTIPLIER
        )

        row.update(centering_metrics(
            image, emission, rectangle, background_level,
            CENTER_TOLERANCE_FRACTION, MASS_FRACTION_REQUIRED
        ))
        row.update({
            "height": image.shape[0],
            "width": image.shape[1],
            "background_level": background_level,
            "background_spread": background_spread,
            "threshold": threshold,
            "emission_fraction": float(emission.mean()),
            "nonfinite_fraction": float((~np.isfinite(image)).mean()),
        })

        if not np.isfinite(image).all():
            row["status"] = "contains nonfinite pixels"

    except Exception as error:
        row["status"] = str(error)

    return row


paths = sorted(DATA_DIR.glob("*.icn.pt"))
if not paths:
    raise FileNotFoundError(f"No *.icn.pt files found in {DATA_DIR}")

if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < len(paths):
    rng = np.random.default_rng(SEED)
    indices = rng.choice(len(paths), size=SAMPLE_LIMIT, replace=False)
    paths = [paths[i] for i in sorted(indices)]

with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=get_context("fork"),
) as pool:
    rows = list(tqdm(
        pool.map(analyze_one, paths),
        total=len(paths),
        desc=f"Analyzing images with {MAX_WORKERS} CPUs",
    ))

results = pd.DataFrame(rows)
for column in (
    "peak_centered", "binary_centered", "weighted_centered",
    "mass_95pct_centered", "mass_fraction_in_rectangle",
):
    if column not in results:
        results[column] = np.nan

print(f"Measured {len(results)} of {len(paths)} selected images.")
display(results.head())

## Compare centered counts


In [ ]:
criteria = {
    "Brightest pixel": "peak_centered",
    "Binary centroid": "binary_centered",
    "Intensity-weighted centroid": "weighted_centered",
    "At least 95% of positive flux in rectangle": "mass_95pct_centered",
}

counts = []
for label, column in criteria.items():
    centered = results[column].eq(True)
    counts.append({
        "criterion": label,
        "centered": int(centered.sum()),
    })

centered_counts = pd.DataFrame(counts)
display(centered_counts)
centered_counts.plot.bar(x="criterion", y="centered", legend=False,
                        ylabel="Number of centered images", rot=30, figsize=(9, 4))
plt.tight_layout()
plt.show()

def select_examples(frame, criteria, pattern, seed, limit):
    columns = list(criteria.values())
    if len(pattern) != len(columns):
        raise ValueError(f"Give {len(columns)} values, one for each criterion")

    selected = frame.copy()
    for column, wanted in zip(columns, pattern):
        if wanted is not None:
            selected = selected.loc[selected[column] == wanted]

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(selected))[:limit]
    selected = selected.iloc[order]
    return selected


## Choose and plot examples

Set `ADD_COLORBAR` and the color limits below. `None` uses the selected percentile
range. `GLOBAL_COLOR_RANGE=True` shares one automatic range across the shown images;
otherwise each image gets its own automatic range. Explicit minimum or maximum values
override the corresponding automatic limit.

In [ ]:
ADD_COLORBAR = True
COLOR_MIN = None
COLOR_MAX = None
GLOBAL_COLOR_RANGE = False
COLOR_PERCENTILES = (0, 100)
cmap = "inferno"

def show_examples(frame, pattern, rectangle_fraction, sigma_multiplier, max_images_per_row,
                  add_colorbar, color_min, color_max, global_color_range, percentiles):
    examples = frame
    if examples.empty:
        print(f"No images match {pattern}.")
        return
    if color_min is not None and color_max is not None and color_min >= color_max:
        raise ValueError("COLOR_MIN must be less than COLOR_MAX")

    ncols = min(max_images_per_row, len(examples))
    nrows = math.ceil(len(examples) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    images = [load_image(row["path"]) for _, row in examples.iterrows()]
    finite_values = [image[np.isfinite(image)] for image in images if np.isfinite(image).any()]
    if global_color_range and finite_values:
        shared_min, shared_max = np.percentile(np.concatenate(finite_values), percentiles)
    else:
        shared_min, shared_max = None, None

    for ax, ((_, row), image) in zip(axes.flat, zip(examples.iterrows(), images)):
        rectangle = rectangular_mask(image.shape, rectangle_fraction)
        background_level, background_spread = estimate_background(image, ~rectangle)
        emission, _ = candidate_mask(image, background_level, background_spread, sigma_multiplier)
        values = image[np.isfinite(image)]
        if values.size:
            own_min, own_max = np.percentile(values, percentiles)
        else:
            own_min, own_max = 0, 1
        low = color_min if color_min is not None else (shared_min if global_color_range else own_min)
        high = color_max if color_max is not None else (shared_max if global_color_range else own_max)

        artist = ax.imshow(image, origin="lower", cmap=cmap, vmin=low, vmax=high)
        if add_colorbar:
            fig.colorbar(artist, ax=ax, fraction=0.046, pad=0.04, label="Intensity")
        if rectangle.any():
            ax.contour(rectangle, levels=[0.5], colors=["yellow"], linewidths=1)
        if emission.any():
            ax.contour(emission, levels=[0.5], colors=["orangered"], linewidths=0.7)
        ax.plot(row["center_x"], row["center_y"], marker="+", color="cyan")
        ax.add_patch(Circle((row["center_x"], row["center_y"]), row["tolerance_px"],
                            fill=False, color="cyan", linewidth=0.7))
        for x, y, marker, color in (
            (row["peak_x"], row["peak_y"], "x", "lime"),
            (row["binary_x"], row["binary_y"], "o", "deepskyblue"),
            (row["weighted_x"], row["weighted_y"], "D", "magenta"),
        ):
            if np.isfinite(x) and np.isfinite(y):
                ax.plot(x, y, marker=marker, color=color, markerfacecolor="none")
        ax.set_title(f"{row['filename']}\n95% flux in rectangle: "
                     f"{row['mass_fraction_in_rectangle']:.2f}")
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in list(axes.flat)[len(examples):]:
        ax.axis("off")

    handles = [
        Line2D([], [], color="yellow", label="Rectangle"),
        Line2D([], [], color="orangered", label="Candidate emission"),
        Line2D([], [], marker="+", color="cyan", linestyle="", label="Center / tolerance"),
        Line2D([], [], marker="x", color="lime", linestyle="", label="Peak"),
        Line2D([], [], marker="o", color="deepskyblue", linestyle="", label="Binary centroid"),
        Line2D([], [], marker="D", color="magenta", linestyle="", label="Weighted centroid"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3)
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    plt.show()

In [ ]:
SHOW_WHERE = [True, False, True, True]
PLOT_SEED = 17
N_PLOT_IMAGES = 6
MAX_IMAGES_PER_ROW = 3


examples = select_examples(results, criteria, SHOW_WHERE, PLOT_SEED, N_PLOT_IMAGES)
print(f"Showing {len(examples)} images matching {SHOW_WHERE}; seed={PLOT_SEED}.")
display(examples[["filename", "peak_centered", "binary_centered", "weighted_centered",
                  "mass_95pct_centered", "mass_fraction_in_rectangle"]])
show_examples(examples, SHOW_WHERE, RECTANGLE_FRACTION, BACKGROUND_SIGMA_MULTIPLIER,
              MAX_IMAGES_PER_ROW, ADD_COLORBAR, COLOR_MIN, COLOR_MAX,
              GLOBAL_COLOR_RANGE, COLOR_PERCENTILES)
